In [0]:
import requests
import pandas as pd

AIR_QUALITY_API_KEY = dbutils.secrets.get(scope ="mlfsbook",key="AIR_QUALITY_API_KEY")
AIR_QUALITY_URL = f"https://api.waqi.info/feed/@2689/?token={AIR_QUALITY_API_KEY}"

response = requests.get(AIR_QUALITY_URL)
data = response.json()

data

In [0]:
date = data["data"]["time"]["iso"]
pm25 = data["data"]["iaqi"]["pm25"]["v"]

date, pm25

In [0]:
from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType
import pandas as pd

schema = StructType([
    StructField("date", TimestampType(), True),
    StructField("pm25", DoubleType(), True)
])

df_air_quality_append = pd.DataFrame([
    {"date": pd.to_datetime(date), "pm25": float(pm25)}
])

spark_df = spark.createDataFrame(
    df_air_quality_append,
    schema=schema
)

spark_df.write.mode("append").saveAsTable("workspace.mlfsbook.lugano_aq_fg")